### Same trasformer as trf.ipynb built with pytorch for tests in Colab.

In [1]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 
import urllib.request 
import os 
import re 
import random 

url = "https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt"
with urllib.request.urlopen(url) as response:
    raw_text = response.read().decode('utf-8')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(device)

if device == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print("Using CPU")


cpu
Using CPU


In [2]:
# Canto headers, e.g. "Inferno: Canto I", "Purgatorio: Canto XXXIII"
canto_header_re = re.compile(r'^(Inferno|Purgatorio|Paradiso):\s*Canto\s+[IVXLCDM]+\s*$')

# Front-matter title lines
title_lines = {
    "LA DIVINA COMMEDIA",
    "di Dante Alighieri",
    "INFERNO",
    "PURGATORIO",
    "PARADISO",
}

def clean_editorial_lines(text):
    cleaned = []
    for line in text.splitlines():
        stripped = line.strip()
        if stripped in title_lines:
            continue
        if canto_header_re.match(stripped):
            continue
        cleaned.append(line)
    return '\n'.join(cleaned)


clean_text = clean_editorial_lines(raw_text)


In [3]:
# Vocabulary 
chars = sorted(list(set(clean_text)))
vocab = len(chars)

stoi = {char : index for index, char in enumerate(chars)}
itos = {index : char for char, index in stoi.items()}

print(stoi.values())
print(itos)

dict_values([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66])
{0: '\n', 1: ' ', 2: '!', 3: '"', 4: "'", 5: '(', 6: ')', 7: ',', 8: '-', 9: '.', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'L', 23: 'M', 24: 'N', 25: 'O', 26: 'P', 27: 'Q', 28: 'R', 29: 'S', 30: 'T', 31: 'U', 32: 'V', 33: 'Z', 34: 'a', 35: 'b', 36: 'c', 37: 'd', 38: 'e', 39: 'f', 40: 'g', 41: 'h', 42: 'i', 43: 'j', 44: 'l', 45: 'm', 46: 'n', 47: 'o', 48: 'p', 49: 'q', 50: 'r', 51: 's', 52: 't', 53: 'u', 54: 'v', 55: 'x', 56: 'y', 57: 'z', 58: '~', 59: 'à', 60: 'è', 61: 'é', 62: 'ì', 63: 'ï', 64: 'ò', 65: 'ó', 66: 'ù'}


# Tokenization and data splitting

In [4]:
def encode(text):
    encoded_text = torch.tensor([stoi[char] for char in text])
    return encoded_text

def decode(text):
    if hasattr(text, 'tolist'):
        text = text.tolist()
    
    decoded_text = [itos[num] for num in text]
    return ''.join(decoded_text)

    # Safety check 


encoded = encode(clean_text)
decoded = decode(encoded)
assert decode(encode(clean_text)) == clean_text

In [5]:
# Data split and batch 
n1 = int(0.9 * len(encoded))
n2 = int(0.95 * len(encoded))

train = encoded[:n1]
val = encoded[n1:n2]
test = encoded[n2:]


def get_batch(split, batch_size, block_size):
    
    if split == 'train':
        data_split = train
    elif split == 'val':
        data_split = val
    elif split == 'test':
        data_split = test
    else:
        raise ValueError('Split must be "train", "val" or "test"')

    ix = torch.randint(0, len(data_split) - block_size, (batch_size,))

    X = torch.stack([
        data_split[index : index + block_size] for index in ix
    ])
    
    Y = torch.stack([
        data_split[ index +1: index + block_size+1] for index in ix
    ])

    return X, Y

# Check 
X, Y = get_batch("train", batch_size=4, block_size=12)

print(X.shape)
print(Y.shape)

print(decode(X[0]))
print(decode(Y[0]))

torch.Size([4, 12])
torch.Size([4, 12])
 suo' mai pe
suo' mai pen


# Transformer architecture


In [6]:
# Building the transformer block:
# Multihead attention, layernorm, and feedforward.

class TransformerBlock(nn.Module):

    def __init__(self, emb_dim, n_head):
        super().__init__()
         
        # Multi-head attention block
        # batch_first=True allows your input tensors to be shaped as (batch, seq, feature)
        self.attention = nn.MultiheadAttention(
            embed_dim=emb_dim, 
            num_heads=n_head, 
            batch_first=True)
    
        # Layer normalisation 
        self.ln1 = nn.LayerNorm(emb_dim)
        self.ln2 = nn.LayerNorm(emb_dim)

        # Feedforward block
        self.feedforward = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4*emb_dim, emb_dim)
        )
    
    def forward(self, x, mask=None):

        # 1. Pre-LN + Self-Attention (same tensor as Q, K, V), causal mask passed in
        norm_x = self.ln1(x)
        attn_out, _ = self.attention(
            query=norm_x,
            key=norm_x, 
            value=norm_x, 
            attn_mask=mask,
            need_weights = False)

        # Residual connection    
        x = x + attn_out
        
        # 2. Pre-LN + Feedforward
        x = x + self.feedforward(self.ln2(x))
        
        return x

In [7]:
# Full language model

class LanguageModel(nn.Module):
    def __init__(self, vocab, emb_dim, block_size, n_head, n_blocks):
        super().__init__()

        self.block_size = block_size

        # Token embeddings
        self.token_emb = nn.Embedding(vocab, emb_dim)
        #Positional embeddings
        self.pos_embeddings = nn.Embedding(block_size, emb_dim)

        # Stacking transformer blocks
        self.n_blocks= nn.ModuleList([TransformerBlock(emb_dim, n_head)
            for _ in range(n_blocks)])

        #Final normalisation
        self.ln_f = nn.LayerNorm(emb_dim)

        # Convert hidden representation into token logits
        self.lm_head = nn.Linear(emb_dim, vocab)

    def forward(self, idx, targets=None):
        batch_size, seq_len = idx.shape

        #Token embeddings
        token_emb = self.token_emb(idx)

        # Positional indices
        positions = torch.arange(seq_len, device= idx.device)
        pos_emb = self.pos_embeddings(positions)

        # Combine token + position information
        x = token_emb + pos_emb

        # Causal maks
        mask = torch.nn.Transformer.generate_square_subsequent_mask(
            seq_len, device=idx.device)

        # Transformer blocks 
        for block in self.n_blocks:
            x = block(x, mask=mask)

        # Final normalization
        x = self.ln_f(x)

        # Vocabulary logits 
        logits = self.lm_head(x)
        loss = None

        if targets is not None:
            # logits:
            # (batch, sequence, vocabulary)
            batch_size, seq_len, vocab_size = logits.shape
            
            # Flatten for cross entropy:
            # (batch * sequence, vocabulary)
            logits = logits.view(batch_size * seq_len, vocab_size)
            
            # Flattening targets
            targets = targets.view(-1) 

            loss = F.cross_entropy(logits, targets)

        return logits, loss

In [8]:
# Hyperparameters (defined once, used everywhere)
batch_size = 64
block_size = 128
emb_dim = 384
n_head = 4
n_blocks = 2

lr = 3e-4
max_steps = 3000
eval_interval = 300
eval_iters = 50

In [9]:
model = LanguageModel(
    vocab=vocab,
    emb_dim=emb_dim,
    block_size=block_size,
    n_head=n_head,
    n_blocks=n_blocks
).to(device)

print(next(model.parameters()).device)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

cpu
3.650371 M parameters


In [10]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}

    for split in ['train', 'val']:
        split_losses = []
        for _ in range(eval_iters):
            xb, yb = get_batch(split, batch_size, block_size)
            xb, yb = xb.to(device), yb.to(device)
            _, loss = model(xb, yb)
            split_losses.append(loss.item())
        losses[split] = sum(split_losses) / len(split_losses)

    model.train()
    return losses

In [11]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for step in range(max_steps):
    xb, yb = get_batch('train', batch_size, block_size)
    xb, yb = xb.to(device), yb.to(device)

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0 or step == max_steps - 1:
        losses = estimate_loss()
        print(
            f"step {step}: "
            f"train {losses['train']:.4f}, "
            f"val {losses['val']:.4f}"
        )

step 0: train 3.8889, val 3.8945
step 300: train 2.0274, val 2.0311
step 600: train 1.7820, val 1.7923
step 900: train 1.6521, val 1.6763
step 1200: train 1.5754, val 1.5975
step 1500: train 1.5241, val 1.5587
step 1800: train 1.4731, val 1.5312
step 2100: train 1.4369, val 1.5082
step 2400: train 1.4014, val 1.5001
step 2700: train 1.3630, val 1.4913
step 2999: train 1.3342, val 1.4848


In [12]:
# Sampling 
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_idx), dim=1)
    model.train()
    return idx

# Start from a single newline character (index 0)
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = generate(model, context, max_new_tokens=500)
print(decode(generated[0]))


d'incominciò sovra le sue dar m'ore;
né d'ogn'io feri dotero a l'altro passo,
  seguemi Litisi 'l voler paese mi mosse!
Malebolienza che l'etalia quattropi mencre,
e quel che n'andar potendi a la pietra,
  per che le prime a le vere al io cammin fosse bella
vota si rugellamente giovate valle
si e a ciò rinverte ne le città discia;
  ond'ellar, più che già li veran "   in percella santa,
malvi nel li antipi Auguto e temperan
  vendo intelle de le spiega nette
in uopere han l'anemo principiensi,
c
